# EZNX-ATLAS-A — Scientific Reports: Complete Training Campaign

**170 experiments** covering primary ablation (3 variants × 20 seeds),  
architecture sensitivity (GLU-width), loss-weight sensitivity (LAUC),  
and data-augmentation sensitivity.

**Run order**: Execute cells 0 → 7 in sequence.  
The notebook is **idempotent**: restart after interruption and it resumes from the last completed experiment.

**Estimated wall time on Kaggle P100**:  
≈ 8–10 min per training run × 170 runs ≈ 24–28 h (2–3 sessions of 12 h each).

**Seeds**: 2024–2043 (consecutive from year of study initiation; no cherry-picking).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 0 — GPU / Environment Verification                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os

print('=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory // 1024**2} MB')

import numpy as np, sklearn
print(f'NumPy    : {np.__version__}')
print(f'sklearn  : {sklearn.__version__}')
print('=== OK ===')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install / upgrade dependencies                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# wfdb is the only package not pre-installed on Kaggle Python images.
# tqdm, scipy, scikit-learn, pandas, pyarrow are pre-installed.
!pip install --quiet wfdb==4.1.2
!pip install --quiet pyarrow>=14.0     # for parquet
print('Dependencies ready.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Path configuration                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ─── EDIT THIS BLOCK ────────────────────────────────────────────────────────
#
# 1. Add the PTB-XL 1.0.3 dataset to this notebook via
#      Notebook settings → Data → Search: "PTB-XL".
#    The recommended dataset is:
#      bjoernjostein/ptb-xl-electrocardiography-dataset
#    After adding it, the data will appear at /kaggle/input/<slug>/
#    Set PTB_XL_SLUG to the folder name that contains ptbxl_database.csv.
#
#    Example: PTB_XL_SLUG = 'ptb-xl-electrocardiography-dataset/1.0.3'
#
PTB_XL_SLUG = 'ptb-xl-electrocardiography-dataset'   # <-- EDIT IF NEEDED
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path

# Locate ptbxl_database.csv within the dataset
KAGGLE_INPUT = Path('/kaggle/input')
WORKING      = Path('/kaggle/working')

# Search for ptbxl_database.csv
import glob
matches = sorted(glob.glob(str(KAGGLE_INPUT / '**' / 'ptbxl_database.csv'), recursive=True))
if not matches:
    raise FileNotFoundError(
        'ptbxl_database.csv not found under /kaggle/input. '
        'Please add the PTB-XL dataset to this notebook.'
    )

PTB_XL_ROOT = Path(matches[0]).parent   # directory containing ptbxl_database.csv
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'

RUNS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f'PTB-XL root : {PTB_XL_ROOT}')
print(f'Index path  : {INDEX_PATH}')
print(f'Runs dir    : {RUNS_DIR}')

# Verify key files exist in PTB-XL root
for fname in ['ptbxl_database.csv', 'scp_statements.csv']:
    p = PTB_XL_ROOT / fname
    assert p.exists(), f'Missing: {p}'
    print(f'  ✓ {fname}')

# Set environment variables for the training scripts
import os
os.environ['EZNX_DATA_REAL']  = str(PTB_XL_ROOT)
os.environ['EZNX_INDEX_PATH'] = str(INDEX_PATH)
os.environ['EZNX_RUNS_DIR']   = str(RUNS_DIR)
print('\nEnvironment variables set.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Build index_complete.parquet                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# This runs index_construction.py which reads ptbxl_database.csv and produces
# index_mm_core.parquet + index_complete.parquet in /kaggle/working/.
#
# Idempotent: skips if index_complete.parquet already exists.

if INDEX_PATH.exists():
    print(f'index_complete.parquet already exists ({INDEX_PATH}) — skipping rebuild.')
else:
    print('Building index …')
    import subprocess, sys
    result = subprocess.run([
        sys.executable, '/kaggle/working/index_construction.py',
        '--data-root', str(PTB_XL_ROOT),
        '--out-dir',   str(WORKING),
    ], capture_output=True, text=True)
    print(result.stdout[-3000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError('index_construction.py failed.')

# Verify
import pandas as pd
idx = pd.read_parquet(INDEX_PATH)
print(f'\nindex_complete.parquet: {len(idx)} rows, {len(idx.columns)} columns')
print('Split distribution:')
print(idx.groupby('strat_fold').size().to_string())

# Integrity checks
required_cols = ['ecg_id', 'patient_id', 'strat_fold', 'scp_codes',
                 'filename_lr', 'filename_hr', 'age_z', 'sex01']
missing = [c for c in required_cols if c not in idx.columns]
assert not missing, f'Missing columns: {missing}'
assert idx['strat_fold'].between(1, 10).all(), 'Unexpected fold values'
print('\nIntegrity checks passed ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Copy uploaded scripts to /kaggle/working/                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Upload the following files via Kaggle's file upload (or attach as a dataset):
#   atlas_a_v5_multiseed_v2.py
#   run_experiments_v2.py
#   compute_calibration.py
#   compute_subgroups.py
#   analyze_multiseed_v2.py
#   evaluate_missingness_v2.py
#   eznx_loader_v2.py
#   eznx_model_v5.py
#   index_construction.py
#
# They should already be at /kaggle/working/ if uploaded via Kaggle.
# This cell verifies all required files are present.

required_scripts = [
    'atlas_a_v5_multiseed_v2.py',
    'run_experiments_v2.py',
    'compute_calibration.py',
    'compute_subgroups.py',
    'analyze_multiseed_v2.py',
    'evaluate_missingness_v2.py',
    'eznx_loader_v2.py',
    'eznx_model_v5.py',
    'index_construction.py',
]

missing_scripts = [s for s in required_scripts if not (WORKING / s).exists()]
if missing_scripts:
    print('\n⚠ MISSING scripts — please upload these files to /kaggle/working/:')
    for s in missing_scripts:
        print(f'  {s}')
    raise FileNotFoundError(f'{len(missing_scripts)} scripts missing.')
else:
    print('All required scripts present ✓')
    for s in required_scripts:
        print(f'  ✓ {s}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Dry run: print the full 170-experiment plan                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys
result = subprocess.run([
    sys.executable, str(WORKING / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_XL_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--dry_run',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Execute all 170 training runs                                ║
# ║                                                                         ║
# ║  IDEMPOTENT: already-complete runs are skipped automatically.           ║
# ║  If the session times out, restart and re-run this cell to continue.   ║
# ║                                                                         ║
# ║  Estimated time on Kaggle P100: ~8–10 min/run × 170 runs ≈ 25 h total ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

# To run a single group only, set --group A | B | C | D
# Default '--group all' runs everything.
result = subprocess.run([
    sys.executable, str(WORKING / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_XL_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'all',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)   # don't raise on failure; progress is saved incrementally

print(f'\nOrchestrator exit code: {result.returncode}')
print(f'Progress CSV: {RESULTS_DIR}/progress.csv')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Post-training evaluation (run AFTER all 170 runs complete)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

print('=== [7a] Statistical analysis (primary + sensitivity) ===')
r = subprocess.run([
    sys.executable, str(WORKING / 'analyze_multiseed_v2.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'stats'),
], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])

print('\n=== [7b] Calibration metrics (Brier + ECE) ===')
r = subprocess.run([
    sys.executable, str(WORKING / 'compute_calibration.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'calibration'),
], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])

print('\n=== [7c] Subgroup AUC analysis (sex / age / missingness) ===')
r = subprocess.run([
    sys.executable, str(WORKING / 'compute_subgroups.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--out_dir',    str(RESULTS_DIR / 'subgroups'),
], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])

print('\n=== [7d] Missingness robustness stress-test ===')
r = subprocess.run([
    sys.executable, str(WORKING / 'evaluate_missingness_v2.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--data_root',  str(PTB_XL_ROOT),
    '--out_dir',    str(RESULTS_DIR / 'missingness'),
], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])

print('\n=== Evaluation complete. ===')

# List all generated result files
import os
for root, dirs, files in os.walk(str(RESULTS_DIR)):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), str(RESULTS_DIR))
        size = os.path.getsize(os.path.join(root, f))
        print(f'  {rel:<60} {size:>10,} B')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Package results for download                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Creates results_package.zip containing:
#   • All results JSON files (one per run) → fast statistical reanalysis
#   • All NPZ probability files → calibration, subgroup, fairness reanalysis
#   • Aggregate summary tables and LaTeX tables
#   • Progress CSV
#
# NOTE: NPZ files may be large (~20 MB each × 170 runs = ~3.4 GB total).
# The JSON-only zip is much smaller and sufficient for most analyses.
#
# Set INCLUDE_NPZ = True to include NPZ files in the zip.
INCLUDE_NPZ = False   # Set True only if you need full probability arrays

import zipfile, os
from pathlib import Path

zip_path = WORKING / 'results_package.zip'
n_files  = 0

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # All JSON results
    for p in sorted(Path(RUNS_DIR).rglob('results_*.json')):
        zf.write(p, p.relative_to(WORKING))
        n_files += 1

    # NPZ probability files (optional)
    if INCLUDE_NPZ:
        for p in sorted(Path(RUNS_DIR).rglob('probs_*.npz')):
            zf.write(p, p.relative_to(WORKING))
            n_files += 1

    # Evaluation outputs
    for p in sorted(RESULTS_DIR.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(WORKING))
            n_files += 1

zip_mb = zip_path.stat().st_size / 1024**2
print(f'\nPackage created: {zip_path}')
print(f'  Files: {n_files}   Size: {zip_mb:.1f} MB')
print(f'\nDownload from Kaggle output panel → results_package.zip')

## After downloading results_package.zip

Send the zip file to the study lead. The following analyses will be run locally
to produce the Scientific Reports manuscript:

1. `analyze_multiseed_v2.py` — primary ablation table, Wilcoxon tests
2. `compute_calibration.py` — Brier / ECE calibration table  
3. `compute_subgroups.py` — sex / age subgroup AUC table  
4. `evaluate_missingness_v2.py` — missingness robustness table  

All scripts are idempotent and self-contained given the JSON / NPZ files.

### Hardware provenance (logged in each results JSON)
- Platform: Kaggle Notebooks  
- GPU: NVIDIA Tesla P100-PCIE-16GB  
- Reproducibility: seeds 2024–2043, CUBLAS deterministic mode, `num_workers=0`